In [1]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
import seaborn as sns
from dython.nominal import associations
from scipy import stats
import sklearn
import os
import time

In [2]:
a = list(range(0,16))
b = [31, 33, 42, 65, 97, 103]
c = list(range(158,667420))
data_cleaned = pd.read_csv("data_1.csv", encoding = "ISO-8859-1", low_memory = False, usecols = a+b+c)


In [3]:
prev_lst_cols = ['Gender (Male / Female)',
 'Ethnicity (Chinese/ Malay/ Indian/ European/ etc)',
 'Primary  Diagnosis ',
 'IMMUNOTHERAPY - 1 -Name of  Immunotherapy ',
 'ICI-induced PULMONARY TOXICITY - Case/Control',
 'ICI-induced HEPATOTOXICITY - Case/Control']

new_lst_cols = ["Gender", "Race", "Cancer", "Immunotherapy drug", "Lung Toxicity", "Liver Toxicity"]
replacing_cols = dict(zip(prev_lst_cols, new_lst_cols))
data_cleaned.rename(columns=replacing_cols, inplace=True)

In [4]:
cols_cleaned = list(data_cleaned.columns)

In [5]:
# Sample Proportions

# print(np.unique(data_cleaned["Lung Toxicity"], return_counts=True))
# print(np.unique(data_cleaned["Liver Toxicity"], return_counts=True))
lung_case = int(np.unique(data_cleaned["Lung Toxicity"], return_counts=True)[1][0])
lung_control = int(np.unique(data_cleaned["Lung Toxicity"], return_counts=True)[1][1])
liver_case = int(np.unique(data_cleaned["Liver Toxicity"], return_counts=True)[1][0])
liver_control = int(np.unique(data_cleaned["Liver Toxicity"], return_counts=True)[1][1])

print(f"Sample proportion of Lung Toxicity: {lung_case/ (lung_control+lung_case)}")
print(f"Sample proportion of Liver Toxicity: {liver_case/ (liver_control+liver_case)}")

Sample proportion of Lung Toxicity: 0.035422343324250684
Sample proportion of Liver Toxicity: 0.04904632152588556


In [6]:
# Gender Proportion

Gender = data_cleaned["Gender"]
gender_dict = {"FEMALE": "F",
              "Female": "F",
              "MALE": "M",
              "Male": "M"}

Gender = Gender.map(gender_dict)
Gender = pd.DataFrame(data = Gender)
Gender.value_counts()

print("Number of male: 257")
print("Number of female: 110")
print(f"Proportion of female: {110/(110+257)}")
print(f"Proportion of male: {257/(110+257)}")

Number of male: 257
Number of female: 110
Proportion of female: 0.2997275204359673
Proportion of male: 0.7002724795640327


In [7]:
data_cleaned.groupby("Liver Toxicity")["Race"].count()

Liver Toxicity
Case        18
Control    349
Name: Race, dtype: int64

In [131]:
cs_con, cs_con_uniques = pd.factorize(data_cleaned["Liver Toxicity"])
size_cs_con_uniques = len(cs_con_uniques)
allele_lst = []
size_allele_uniques_lst = []
array_all = []
allele_uniques_lst = []
for alleles in range(24, 667283): 
    allele, allele_uniques = pd.factorize(data_cleaned[cols_cleaned[alleles]])
    size_allele_uniques = len(allele_uniques)
    allele_lst.append(allele)
    size_allele_uniques_lst.append(size_allele_uniques)
    allele_uniques_lst.append(allele_uniques)

for count, things in enumerate(size_allele_uniques_lst):
    array_iteration = np.zeros((size_cs_con_uniques, things), dtype=np.int64)
    np.add.at(array_iteration, (cs_con, allele_lst[count]), 1)
    array_all.append(array_iteration)


In [171]:
modified_lst = []
for val in allele_uniques_lst:
    stuff = val.tolist()
    modified_lst.append(stuff)
    
array_all_modified = []
for val in array_all:
    stuff = val.tolist()
    array_all_modified.append(stuff)


for count, alleles in enumerate(modified_lst):
    if "?_?" in alleles:
        index_of_unknown = alleles.index("?_?")
        del array_all_modified[count][0][index_of_unknown]
        del array_all_modified[count][1][index_of_unknown]
        


In [184]:
three_stuff = []
for matters in array_all_modified:
    if len(matters[0]) == 3:
        three_stuff.append(matters)

In [186]:
len(three_stuff)

315015

In [187]:
two_stuff = []
for matters in array_all_modified:
    if len(matters[0]) == 2:
        two_stuff.append(matters)
len(two_stuff)

172462

In [188]:
one_stuff = []
for matters in array_all_modified:
    if len(matters[0]) == 1:
        one_stuff.append(matters)
len(one_stuff)

179593

In [189]:
len(three_stuff) + len(two_stuff) + len(one_stuff)

667070

In [133]:
lst_without_unknowns = []
for count, alleles in enumerate(allele_uniques_lst):
    if "?_?" in alleles:
        continue
    else:
        lst_without_unknowns.append(count)
len(lst_without_unknowns)

578380

In [193]:
possible_hits_for_two_stuff = []
for stuff in two_stuff:
    try:
        if (stuff[0][0]/stuff[0][1] > 1 and stuff[1][0]/stuff[1][1] < 1) or (stuff[0][0]/stuff[0][1] < 1 and stuff[1][0]/stuff[1][1] > 1):
            possible_hits_for_two_stuff.append(stuff)
    except:
        continue

In [194]:
possible_hits_for_two_stuff

[[[139, 204], [10, 7]],
 [[129, 210], [9, 8]],
 [[160, 177], [11, 7]],
 [[130, 148], [8, 6]],
 [[126, 154], [12, 6]],
 [[140, 145], [8, 6]],
 [[90, 194], [8, 7]],
 [[83, 116], [7, 4]],
 [[117, 148], [9, 5]],
 [[127, 140], [8, 5]],
 [[140, 130], [4, 5]],
 [[154, 176], [9, 8]],
 [[147, 186], [9, 7]],
 [[168, 177], [9, 7]],
 [[144, 187], [10, 6]],
 [[151, 123], [5, 11]],
 [[185, 134], [7, 8]],
 [[122, 159], [7, 5]]]

In [125]:
potential_hits = []
for stuff in array_all:
    if len(stuff[1]) != 2:
        continue
    else:
        if sorted(stuff[1])[0]/sorted(stuff[1])[1] > 0.3:
            potential_hits.append(stuff)
potential_hits

[array([[244, 105],
        [  9,   9]], dtype=int64),
 array([[265,  84],
        [ 13,   5]], dtype=int64),
 array([[132, 217],
        [  8,  10]], dtype=int64),
 array([[166, 183],
        [  8,  10]], dtype=int64),
 array([[293,  56],
        [ 13,   5]], dtype=int64),
 array([[154, 195],
        [  7,  11]], dtype=int64),
 array([[255,  94],
        [ 11,   7]], dtype=int64),
 array([[177, 172],
        [  9,   9]], dtype=int64),
 array([[242, 107],
        [ 11,   7]], dtype=int64),
 array([[180, 169],
        [ 13,   5]], dtype=int64),
 array([[175, 174],
        [ 11,   7]], dtype=int64),
 array([[280,  69],
        [ 12,   6]], dtype=int64),
 array([[ 56, 293],
        [  5,  13]], dtype=int64),
 array([[315,  34],
        [ 13,   5]], dtype=int64),
 array([[258,  91],
        [  8,  10]], dtype=int64),
 array([[272,  77],
        [  9,   9]], dtype=int64),
 array([[155, 194],
        [ 11,   7]], dtype=int64),
 array([[196, 153],
        [  9,   9]], dtype=int64),
 array([[2

In [128]:
potential_hits[14]

array([[258,  91],
       [  8,  10]], dtype=int64)

In [117]:
potential_hits = []
for stuff in array_all[0:200]:
    if len(stuff[1]) == 3:
        continue
    else:
        if sorted(stuff[1])[0]/sorted(stuff[1])[1] > 0.01:
            potential_hits.append(stuff)
potential_hits 

[array([[347,   2],
        [ 17,   1]], dtype=int64),
 array([[335,  14],
        [ 17,   1]], dtype=int64),
 array([[  5, 344],
        [  1,  17]], dtype=int64),
 array([[346,   3],
        [ 17,   1]], dtype=int64),
 array([[348,   1],
        [ 17,   1]], dtype=int64),
 array([[345,   4],
        [ 17,   1]], dtype=int64)]

In [110]:
potential_hits = []
for stuff in array_all:
    if stuff[1] != 2:
        continue
    else:
        if sorted(stuff[1][0])/sorted(stuff[1][1]) > 0.2:
            potential_hits.append(stuff)
potential_hits         

ValueError: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()

In [38]:
# print(data_cleaned.groupby(['Liver Toxicity', cols_cleaned[24]]).agg({cols_cleaned[24]: "count"}))

                               chr1-100350260
Liver Toxicity chr1-100350260                
Case           G_G                         18
Control        ?_?                          1
               G_G                        348


In [10]:
# case_lung = data_cleaned[data_cleaned["Lung Toxicity"] == "Case"]
# control_lung = data_cleaned[data_cleaned["Lung Toxicity"] == "Control"]
# case_liver = data_cleaned[data_cleaned["Liver Toxicity"] == "Case"]
# control_liver = data_cleaned[data_cleaned["Liver Toxicity"] == "Control"]

In [11]:
# significant_alleles = []
# for alleles in range(24, 667000):
#     if len(np.unique(data_cleaned[cols_cleaned[alleles]], return_counts=True)[1]) == 2:
#         if sorted(np.unique(data_cleaned[cols_cleaned[alleles]], return_counts=True)[1])[0]/sorted(np.unique(data_cleaned[cols_cleaned[alleles]], return_counts=True)[1])[1] > 0.4:
#             significant_alleles.append(alleles)
        

In [12]:
# genotype = []
# hits_lst = []
# hits_genotype_lst = []
# for alleles in significant_alleles:
#     genotype.append(np.unique(data_cleaned[cols_cleaned[alleles]], return_counts=True)[0])
# for position, alleles in enumerate(genotype):
#     if alleles[0] == "?_?":
#         continue
#     else:
#         hits_lst.append(position)
        
# for hits in hits_lst:
#     hits_genotype_lst.append(np.unique(data_cleaned[cols_cleaned[significant_alleles[hits]]], return_counts=True))

In [13]:
# lst_of_hits_pos = []
# for hits in hits_lst:
#     lst_of_hits_pos.append(significant_alleles[hits])

In [33]:
b = np.zeros((2,3), dtype=np.int64)

np.add.at(b, ([2,3,3,4,6,8,5,7], [2,5,7,4,8,7,7,3]), 1)
b

array([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 1, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 1, 0, 1, 0, 0],
       [0, 0, 0, 0, 1, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 1, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 1, 0],
       [0, 0, 0, 1, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 1, 0, 0]], dtype=int64)

In [8]:
# lst_of_codes = []
# lst_of_uniques = []
# for alleles in range(24, 667283):
#     codes, uniques = pd.factorize(data_cleaned[cols_cleaned[alleles]])
#     lst_of_codes.append(codes)
#     lst_of_uniques.append(uniques)

In [16]:
# for i in hits_genotype_lst:
#     print(i[0])

In [17]:
# count_lst = []
# for i, hits in enumerate(lst_of_hits_pos):
#     counts =len(data_cleaned[(data_cleaned['Lung Toxicity'] == 'Case') & (data_cleaned[cols_cleaned[hits] == hits_genotype_lst[i][0][0]])])

In [18]:
data_cleaned.groupby(["Race", "Liver Toxicity"])["Liver Toxicity"].count()

Race       Liver Toxicity
?          Control             1
Burmese    Control             1
Caucasian  Case                1
           Control             1
Chinese    Case               15
           Control           267
Eurasian   Control             1
Indian     Control            13
Malay      Case                2
           Control            46
Others     Control            18
Sikh       Control             1
Name: Liver Toxicity, dtype: int64

In [19]:
data_cleaned.groupby(["Immunotherapy drug", "Liver Toxicity"])["Liver Toxicity"].count()

Immunotherapy drug                                                                            Liver Toxicity
?                                                                                             Case                1
                                                                                              Control             8
AB122 (Zimberelimab)                                                                          Case                1
ATEZOLIZUMAB                                                                                  Control             1
Atezolimumab                                                                                  Control             1
Atezolizumab                                                                                  Case                4
                                                                                              Control            32
Atezolizumab (MPDL)                                                            

In [20]:
# mask = data_cleaned[data_cleaned['Race'] == 'Chinese']
# chinese_data = data_cleaned[mask]
# chinese_data[cols_cleaned[1]]

In [21]:
# import pandas as pd
# codes, uniques = pd.factorize(['A_A', 'A_G', 'A_A', 'A_G', 'A_G'])
# codes
# uniques

In [22]:
# for pos in lst_of_hits_pos:
#     print(data_cleaned.groupby(['Lung Toxicity', cols_cleaned[pos]]).agg({cols_cleaned[pos]: "count"}))